# B07-04 — 배지 성분 중요도·분포 이동·모델 불확실성 감사

**BioCulture | 실제 *E. coli* 배양 자료 | 실행 전용 Notebook v1**

B07-00의 실제 조성 식별자와 B07-02/03의 검증 결과를 계승한다. 핵심 질문: 내부 분할에서는 잘 맞던 모델이 새 라운드에서 왜 실패하는가? 중요한 성분은 얼마나 안정적인가?

**경계:** 입력은 배지 성분 44개와 원본 연구의 성장 지표 `K_median`, `r_median`. 상관·중요도는 인과효과가 아니다. 반복 모델의 예측 폭은 보정된 예측구간이 아니다. 원본 `K/r`의 생물학적 정의는 별도 확인이 필요하다.

## 1. 의존성 및 실행 환경

필요 패키지: `numpy`, `pandas`, `scikit-learn`, `matplotlib`. 저장된 B07-00·B07-03 산출물을 검사하며, B07-02에서 결정한 탐색적 RF 설정을 **고정**하여 비교한다. B07-03의 라운드 분할을 정확히 재구성한다.

In [ ]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cwd=Path.cwd().resolve()
ROOT=next((p for p in (cwd,cwd.parent,cwd/'bioimage-ai') if (p/'results'/'bioculture'/'B07_00_ml_data_audit_v2').is_dir()),None)
if ROOT is None: raise FileNotFoundError('프로젝트 루트 또는 notebooks에서 실행해야 합니다.')
BASE=ROOT/'results'/'bioculture'
A=BASE/'B07_00_ml_data_audit_v2'
G=BASE/'B07_03_group_round_validation_v1'
OUT=BASE/'B07_04_interpretability_uncertainty_v1';OUT.mkdir(parents=True,exist_ok=True)
PROFILE=A/'ml_medium_profile_level_audit_table.csv'
FEATURES=A/'medium_feature_audit.csv'
CURVES=A/'ml_curve_level_audit_table.csv'
REP=G/'repeated_group_holdout_predictions.csv'
ROUNDP=G/'leave_one_round_out_predictions.csv'
ROUND_DESIGN=G/'leave_one_round_out_design.csv'
for path in (PROFILE,FEATURES,CURVES,REP,ROUNDP,ROUND_DESIGN):
    if not path.is_file():raise FileNotFoundError(path)
print('Python:',sys.version.split()[0],'프로젝트:',ROOT)

## 2. 데이터 구조와 배지 조성 단위 감사

행을 13,608개 곡선으로 부풀리지 않고 고유 조성 1,025개를 사용한다. 조성의 실험 라운드 소속은 별도 표로 보존한다.

In [ ]:
profile=pd.read_csv(PROFILE,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
fa=pd.read_csv(FEATURES,encoding='utf-8-sig')
feature_cols=fa['feature'].astype(str).tolist()
assert len(feature_cols)==44 and len(set(feature_cols))==44
assert len(profile)==1025 and profile.medium_profile_id.is_unique
X=profile.set_index('medium_profile_id')[feature_cols].apply(pd.to_numeric,errors='coerce')
ytab=profile.set_index('medium_profile_id')[['K_median','r_median']].apply(pd.to_numeric,errors='coerce')
assert X.notna().all().all() and X.index.is_unique
meta=pd.read_csv(CURVES,encoding='utf-8-sig',usecols=['medium_profile_id','round'],dtype={'medium_profile_id':'string'})
meta['round_key']=meta['round'].map(lambda v: str(int(float(v))) if pd.notna(v) and str(v).replace('.','',1).isdigit() else str(v))
round_sets=meta.groupby('medium_profile_id')['round_key'].agg(lambda s:tuple(sorted(set(s)))).reindex(X.index)
assert round_sets.notna().all()
round_design=pd.read_csv(ROUND_DESIGN,encoding='utf-8-sig')
round_pred=pd.read_csv(ROUNDP,encoding='utf-8-sig')
repeat_pred=pd.read_csv(REP,encoding='utf-8-sig')
print('조성:',X.shape,'성장목표 유효 수:',ytab.notna().sum().to_dict())
print('라운드 예측 컬럼:',list(round_pred.columns))
print('반복 예측 컬럼:',list(repeat_pred.columns))

## 3. Round 02 및 05 실패 진단

B07-03에서 저장한 실제 예측값·관측값을 대상으로 범위, 평균, 표준편차, MAE, RMSE와 R²를 재계산한다. R²가 지나치게 음수일 때 시험 집합의 `y_true` 변동이 작은지 우선 확인한다.

In [ ]:
assert {'held_out_round','target','model','y_true','y_pred'}.issubset(round_pred.columns), list(round_pred.columns)
rows=[]
for (r,target,model),q in round_pred.groupby(['held_out_round','target','model']):
    if model!='RandomForest':continue
    a=q.y_true.to_numpy(dtype=float);b=q.y_pred.to_numpy(dtype=float)
    if len(a)<2:continue
    rows.append({'round':str(r),'target':target,'n':len(a),'observed_mean':np.mean(a),
       'observed_sd':np.std(a,ddof=1),'predicted_mean':np.mean(b),
       'bias_mean_pred_minus_true':np.mean(b-a),'MAE':mean_absolute_error(a,b),
       'RMSE':np.sqrt(mean_squared_error(a,b)),'R2':r2_score(a,b)})
fail=pd.DataFrame(rows)
fail.to_csv(OUT/'round_failure_diagnostics.csv',index=False,encoding='utf-8-sig')
print(fail[fail['round'].isin(['2','5'])].round(5).to_string(index=False))

## 4. 학습 조성 대비 입력 분포 이탈

각 라운드에서 해당 라운드에 등장한 모든 조성은 학습에서 제외한다. 시험에는 **그 라운드에만 등장하는 조성**을 사용한다. 학습 데이터의 성분별 최솟값·최댓값을 넘어가는 비율 및 표준화된 최근접거리의 중앙값을 탐색적으로 계산한다. 이 지표들은 분포 이동의 **대용 지표**이며 원인 증명이 아니다.

In [ ]:
shift_rows=[]
for r in sorted(set(meta.round_key)):
    seen=[pid for pid,rs in round_sets.items() if r in rs]
    pure=[pid for pid in seen if len(round_sets.loc[pid])==1]
    train=[pid for pid,rs in round_sets.items() if r not in rs]
    assert not set(train)&set(seen)
    if not pure or not train:continue
    tr=X.loc[train];te=X.loc[pure]
    lo=tr.min();hi=tr.max()
    outside=te.lt(lo,axis='columns')|te.gt(hi,axis='columns')
    scale=tr.std(ddof=0).replace(0,1)
    # 과도한 메모리 사용을 피하도록 한 시험 조성씩 계산
    nearest=[]
    T=((tr-tr.mean())/scale).to_numpy(dtype=float)
    for row in ((te-tr.mean())/scale).to_numpy(dtype=float):
        nearest.append(float(np.sqrt(((T-row)**2).sum(axis=1)).min()))
    shift_rows.append({'held_out_round':r,'n_train':len(train),'n_test_pure':len(pure),
      'share_test_profiles_with_any_out_of_train_range':float(outside.any(axis=1).mean()),
      'mean_share_features_outside_train_range':float(outside.to_numpy().mean()),
      'median_nearest_scaled_distance':float(np.median(nearest))})
shift=pd.DataFrame(shift_rows)
shift.to_csv(OUT/'round_feature_shift_audit.csv',index=False,encoding='utf-8-sig')
print(shift.round(4).to_string(index=False))

## 5. Per​mutation Importance와 분할별 안정성

B07-02의 80/20 조성 단위 분할을 3개 시드에서 재구성하고, 훈련에서만 학습한 Random Forest를 시험 조성에 평가한다. **Permutation Importance는 시험 조성의 R² 감소량**으로 측정한다. 상관된 성분은 중요도를 서로 나누거나 불안정하게 만들 수 있다. 세 시드×각 목표의 결과는 안정성 탐색이며 독립 실험 반복이 아니다.

In [ ]:
SEEDS=[20261019,20261020,20261021]
TARGETS=['K_median','r_median']
MAXF={'K_median':1.0,'r_median':0.5}
importance_rows=[]; heldout_rows=[]; ensemble_predictions=[]
for target in TARGETS:
    for seed in SEEDS:
        tr_ids,te_ids=train_test_split(X.index.astype(str).tolist(),test_size=0.2,random_state=seed,shuffle=True)
        tr_ids=[v for v in tr_ids if pd.notna(ytab.loc[v,target])]
        te_ids=[v for v in te_ids if pd.notna(ytab.loc[v,target])]
        assert not set(tr_ids)&set(te_ids)
        m=RandomForestRegressor(n_estimators=120,max_features=MAXF[target],min_samples_leaf=2,
                               n_jobs=-1,random_state=seed)
        m.fit(X.loc[tr_ids],ytab.loc[tr_ids,target])
        pred=m.predict(X.loc[te_ids]);truth=ytab.loc[te_ids,target].to_numpy()
        heldout_rows.append({'seed':seed,'target':target,'n_test':len(te_ids),
                             'R2':r2_score(truth,pred),'MAE':mean_absolute_error(truth,pred)})
        pi=permutation_importance(m,X.loc[te_ids],ytab.loc[te_ids,target],n_repeats=3,
                                  scoring='r2',random_state=seed,n_jobs=1)
        for col,v,sd in zip(feature_cols,pi.importances_mean,pi.importances_std):
            importance_rows.append({'seed':seed,'target':target,'feature':col,'importance_r2_drop':v,
                                    'permutation_sd':sd})
        # 하나의 고정 시험 분할에 대해 같은 훈련 자료를 bootstrap 재학습하여 분산을 산출하기 위함
        if seed==SEEDS[0]:
            fixed_train=list(tr_ids);fixed_test=list(te_ids)
            rng=np.random.default_rng(seed)
            for b in range(12):
                sampled=rng.choice(fixed_train,size=len(fixed_train),replace=True)
                boot=RandomForestRegressor(n_estimators=70,max_features=MAXF[target],min_samples_leaf=2,
                                          n_jobs=-1,random_state=seed+b+100)
                boot.fit(X.loc[sampled],ytab.loc[sampled,target])
                values=boot.predict(X.loc[fixed_test])
                ensemble_predictions.extend([{'target':target,'medium_profile_id':pid,'bootstrap':b,'prediction':float(val)}
                                             for pid,val in zip(fixed_test,values)])
importance=pd.DataFrame(importance_rows)
importance.to_csv(OUT/'permutation_importance_by_split.csv',index=False,encoding='utf-8-sig')
score=pd.DataFrame(heldout_rows)
score.to_csv(OUT/'importance_holdout_scores.csv',index=False,encoding='utf-8-sig')
imp_summary=importance.groupby(['target','feature'],as_index=False).agg(
    mean_importance=('importance_r2_drop','mean'),sd_across_splits=('importance_r2_drop','std'),
    positive_splits=('importance_r2_drop',lambda x:int((x>0).sum())))
imp_summary.to_csv(OUT/'permutation_importance_stability.csv',index=False,encoding='utf-8-sig')
for target in TARGETS:
    print('\n',target,'상위 중요 성분 (인과효과 아님)')
    display(imp_summary[imp_summary.target==target].nlargest(8,'mean_importance').round(4))

## 6. Bootstrap 모델 간 예측 불일치

동일한 훈련 세트에서 조성 행을 복원추출하고 Random Forest를 12회 재학습한다. 각 시험 조성에 대한 예측 표준편차·10~90% 분위수 폭을 산출한다. 이는 **모델 재학습에 따른 예측 불안정성**이지 관측 노이즈, 새로운 라운드 변화, 진정한 조건부 예측 불확실성 전부를 포착한 것은 아니다. 보정된 예측구간으로 제시하지 않는다.

In [ ]:
boot_pred=pd.DataFrame(ensemble_predictions)
spread=boot_pred.groupby(['target','medium_profile_id'],as_index=False).agg(
  model_pred_mean=('prediction','mean'),model_pred_sd=('prediction','std'),
  q10=('prediction',lambda s:s.quantile(.10)),q90=('prediction',lambda s:s.quantile(.90)))
spread['p80_model_spread']=spread.q90-spread.q10
spread['actual']=spread.apply(lambda row:ytab.loc[row.medium_profile_id,row.target],axis=1)
spread['absolute_error']=abs(spread.model_pred_mean-spread.actual)
spread.to_csv(OUT/'bootstrap_model_disagreement.csv',index=False,encoding='utf-8-sig')
print(spread.groupby('target')[['model_pred_sd','p80_model_spread','absolute_error']].median().round(5))
print('모델 예측 10~90% 범위는 80% 관측 예측구간이 아닙니다.')

## 7. 진단 그래프와 결과 검증

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for target in TARGETS:
    q=fail[fail.target==target]
    axes[0].plot(q['round'],q.R2,marker='o',label=target)
axes[0].axhline(0,color='gray',lw=.8);axes[0].set(title='RF held-out rounds',xlabel='Round',ylabel='R²');axes[0].legend()
axes[1].plot(shift.held_out_round,shift.mean_share_features_outside_train_range,marker='o')
axes[1].set(title='Features outside training range',xlabel='Round',ylabel='Feature fraction')
for target in TARGETS:
    q=imp_summary[imp_summary.target==target].nlargest(6,'mean_importance').sort_values('mean_importance')
    axes[2].barh([f'{target}: {s}' for s in q.feature],q.mean_importance,alpha=.65)
axes[2].set(title='Permutation importance (exploratory)',xlabel='Mean heldout R² drop')
fig.tight_layout();fig.savefig(OUT/'interpretability_shift_overview.png',dpi=150,bbox_inches='tight');plt.show()
assert len(imp_summary)==88
assert set(spread.medium_profile_id).issubset(set(X.index.astype(str)))
for f in ['round_failure_diagnostics.csv','round_feature_shift_audit.csv','permutation_importance_stability.csv','bootstrap_model_disagreement.csv']:
    assert len(pd.read_csv(OUT/f,encoding='utf-8-sig'))>0
print('저장 후 재읽기 검증 통과')

## 8. 재현성 보고서 및 감사 경계

In [ ]:
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
report={'status':'INTERPRETABILITY_AND_SHIFT_AUDITED_EXPLORATORY_NOT_CAUSAL',
  'n_profiles':int(len(X)),'n_features':int(len(feature_cols)),
  'targets':TARGETS,'importance_seeds':SEEDS,'bootstrap_refits_per_target':12,
  'round_holdout_from':'B07-03 saved out-of-round pure-profile predictions',
  'critical_limits':['Round shift and medium composition shift are confounded',
     'Permutation importance is not causal importance and correlated features complicate attribution',
     'Bootstrap refit prediction spread is not a calibrated prediction interval',
     'B07-02 hyperparameters fixed from exploratory tuning; not independent nested model selection',
     'K/r original scientific interpretation not independently verified']}
(OUT/'B07_04_validation_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
manifest=[{'kind':'input','name':p.name,'sha256':sha(p)} for p in (PROFILE,FEATURES,CURVES,REP,ROUNDP,ROUND_DESIGN)]
manifest += [{'kind':'output','name':p.name,'sha256':sha(p)} for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='file_sha256_manifest.csv']
pd.DataFrame(manifest).to_csv(OUT/'file_sha256_manifest.csv',index=False,encoding='utf-8-sig')
print('상태:',report['status'],'산출물:',len(list(OUT.iterdir())))

## 9. 학습 정리 및 연습문제

1. Round 02의 `r_median`에서 R²가 매우 음수인 현상을 실제값의 표준편차와 RMSE로 설명해 보라.
2. 왜 배지 성분 간 높은 상관성이 있을 때 Permutation Importance 순위가 불안정해질 수 있는가? 두 시드의 순위를 비교하는 코드를 작성하라.
3. Round 05에서 훈련 성분 범위를 벗어난 시험 조성의 비율을 구하라. 이 값만으로 실험 라운드의 영향을 입증할 수 있는가?
4. `p80_model_spread`가 넓은 조성 10개를 찾고 오차와 나란히 비교하라. 이 폭을 80% 신뢰구간이라고 부를 수 있는가?
5. 같은 `medium_profile_id`가 훈련·시험에 동시에 들어가지 않아야 하는 이유를 설명하고, 관련 assert를 작성하라.

**정답과 Python 풀이**는 별도의 B07-04 Word 해설서에 수록한다. 본 Notebook은 실행 결과와 연구적 질문을 분리한다.